In [ ]:
import os

def get_matching_files(directory, pattern):
    """
    Returns a list of file paths in 'directory' whose names contain 'pattern'.
    """
    return [
        os.path.join(directory, fname)
        for fname in os.listdir(directory)
        if pattern in fname
    ]

In [ ]:
# fetch log files for evaluation
import re
import polars as pl

# use a looser matcher so files like:
#  - dt_rtg03_test_episode_02_logs.parquet
#  - dt_test_episode_02_rtg03_logs.parquet
# are both returned
matching_files = get_matching_files("../data", "test_episode_02")

# capture algorithm at start, trailing _logs (allow optional .parquet)
pat = re.compile(r"^(?P<algo>[A-Za-z0-9]+).*?_logs(?:\.parquet)?$", re.IGNORECASE)
# find _rtgNN anywhere in the name
rtg_search_re = re.compile(r"_rtg(?P<rtg>\d+)", re.IGNORECASE)

test_logs: dict[str, pl.DataFrame] = {}
skipped = []

for file_path in matching_files:
    fname = os.path.basename(file_path)
    m = pat.match(fname)
    if not m:
        print(f"Skipping unrecognized filename: {fname}")
        skipped.append(fname)
        continue

    algo = m.group("algo")
    rtg_m = rtg_search_re.search(fname)
    rtg = rtg_m.group("rtg") if rtg_m else None
    label = f"{algo}_rtg{rtg}" if rtg else algo

    # try to read parquet; if unreadable or empty, register empty DF so label exists
    try:
        if os.path.getsize(file_path) == 0:
            df = pl.DataFrame()
            print(f"Warning: zero-byte file -> empty DF for {label}: {fname}")
        else:
            df = pl.read_parquet(file_path)
    except Exception as e:
        print(f"Failed to read {fname}: {e} -> registering empty DF for {label}")
        df = pl.DataFrame()

    # ensure episode_id exists for non-empty frames
    if not df.is_empty() and "episode_id" not in df.columns:
        df = df.with_columns(pl.lit(0).alias("episode_id"))

    test_logs[label] = df
    print(f"Registered: {fname} -> label='{label}', rows={len(df)}")

print("Final labels:", list(test_logs.keys()))
print("Skipped files:", skipped)

In [ ]:
from helper import evaluate_experiments
# Split into a list of DataFrames, one per episode
all_logs = {
    label: [df.filter(pl.col("episode_id") == eid) for eid in df["episode_id"].unique()]
    for label, df in test_logs.items()
}

# evaluate experiments outcomes
metrics = evaluate_experiments(all_logs, target_return=0.0)
print(metrics)

In [ ]:
for algo, logs in all_logs.items():
    print(f"{algo}: {len(logs)} episodes")

In [ ]:
from helper import AlgorithmActionComparator, ActionComparisonConfig
import matplotlib.pyplot as plt
# time periods to compare in steps
# first 2 days, 2 days after 4th weeks, 2 days after 4th months, 2 days at 11th months (assuming 30-min per step, 48 steps/day)
first_time_mark = 0
second_time_mark = 4 * 7 * 48  # 4 weeks
third_time_mark = 4 * 30 * 48  # 4 months
fourth_time_mark = 11 * 30 * 48  # 11 months
two_day_steps = 2 * 48

time_periods = [
    (first_time_mark, first_time_mark + two_day_steps),
    (second_time_mark, second_time_mark + two_day_steps),
    (third_time_mark, third_time_mark + two_day_steps),
    (fourth_time_mark, fourth_time_mark + two_day_steps),
]

# pick 52nd episode if exists for all algorithms
episode_52 = {}
for algo, logs in all_logs.items():
    if len(logs) > 52:
        episode_52[algo] = logs[52]

# Wrap single-episode DataFrames into lists expected by the comparator
episode_single = {k: [v] for k, v in episode_52.items()}

# check if we have data for episode 52
if not episode_52:
    print("No data available for episode 52 across the algorithms.")

In [ ]:
# Create comparator and config, then run
comparator = AlgorithmActionComparator()
cfg = ActionComparisonConfig(time_periods=time_periods, return_figs=True)
result = comparator.compare(logs_dict=episode_single, config=cfg)

# Extract metrics and figures (the comparator returns an ActionComparisonResult when return_figs=True)
if hasattr(result, 'metrics'):
    metrics = result.metrics
    figs = result.figures
else:
    metrics = result
    figs = {}

# Display metrics and figures
print(metrics)
for fig in figs.values():
    plt.show(fig)

In [ ]:
from helper import AlgorithmActionComparator, TemporalAnalysisConfig
import matplotlib.pyplot as plt

# Use the same time_periods defined earlier and the episode_8964 mapping (algo -> DataFrame)
# episode_8964 maps algorithm -> single episode DataFrame; analyze_temporal expects dict[str, pl.DataFrame]

comparator = AlgorithmActionComparator()
cfg = TemporalAnalysisConfig(time_periods=time_periods, annotate_states=True, step_duration=0.5, reference=None, action_tolerance=0.01)

result = comparator.analyze_temporal(logs_dict=episode_52, config=cfg)
fig = result.figure
stats = result.stats

# Display stats and figure
print(stats)
# In notebooks, use display of Matplotlib figure
plt.show(fig)


In [ ]:
# Example: Evaluate rewards under different conditions for one algorithm
from helper import evaluate_by_conditions

# Define your conditions (functions that take obs and return True/False)
conditions = {
    "high_solar": lambda obs: obs[5] > 2.0,
    "peak_price": lambda obs: obs[7] > 0.2,
    "low_battery": lambda obs: obs[-2] < 0.3
}

# logs is a list of Polars DataFrames for one algorithm
results = evaluate_by_conditions(logs, conditions)
print(results)